In [2]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datetime import datetime
from urllib import response
from aisuite import Client
from src.research_tools import (
    arxiv_search_tool,
    tavily_search_tool,
    wikipedia_search_tool,
)
from typing import List, Dict, Optional

import json
import xml.etree.ElementTree as ET
from io import BytesIO
from src.agents import key_word, research_agent
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from typing import List, Dict, Optional
import os
import re
import time
import tempfile
import xml.etree.ElementTree as ET
from io import BytesIO
import wikipedia

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
client = Client()

In [ ]:
key_words = key_word('meu plano é criar um novo algorítimo igual ao transformer, então gostaria de entender artigos que buscam desenvolver algorítimos recentes sobre IA')
key_words

In [3]:
key_words = {'search_phrases': ['recent AI algorithms',
  'transformer model development',
  'novel deep learning architectures',
  'advancements in neural networks',
  'state-of-the-art AI models',
  'transformer alternatives',
  'machine learning algorithm innovation',
  'AI algorithm design 2023',
  'transformer improvements',
  'emerging AI architectures'],
 'url_to_query': 'recent AI algorithms or transformer model development or novel deep learning architectures or advancements in neural networks or state-of-the-art AI models or transformer alternatives or machine learning algorithm innovation or AI algorithm design 2023 or transformer improvements or emerging AI architectures'}

In [4]:
import importlib
import src.research_tools as research_tools

importlib.reload(research_tools)

papers = research_tools.arxiv_search_tool(
    key_words.get("url_to_query"),
    max_results=4
)

Starting arXiv request...


In [5]:
id_list = [p.get("arxiv_id") for p in papers]

papers

[{'error': 'arXiv API request failed: 429 Client Error: Unknown Error for url: https://export.arxiv.org/api/query?search_query=all:recent%20AI%20algorithms%20or%20transformer%20model%20development%20or%20novel%20deep%20learning%20architectures%20or%20advancements%20in%20neural%20networks%20or%20state-of-the-art%20AI%20models%20or%20transformer%20alternatives%20or%20machine%20learning%20algorithm%20innovation%20or%20AI%20algorithm%20design%202023%20or%20transformer%20improvements%20or%20emerging%20AI%20architectures&start=0&max_results=4'}]

In [ ]:
import requests
import time
import re

def is_valid_arxiv_id(arxiv_id: str) -> bool:
    """
    Valida IDs do arXiv no formato moderno:
    YYMM.NNNN ou YYMM.NNNNN
    Ex.: 1706.03762, 2501.02842
    """
    arxiv_id = arxiv_id.strip()
    pattern = r"^\d{4}\.\d{4,5}$"
    return bool(re.match(pattern, arxiv_id))


def get_citation_counts_batch(id_list, batch_size=100, sleep_seconds=3):
    """
    Busca citationCount no Semantic Scholar usando o endpoint batch.

    Retorna uma lista de dicionários com:
    - arxiv_id
    - citationCount
    - status
    """

    url = "https://api.semanticscholar.org/graph/v1/paper/batch"
    params = {"fields": "citationCount"}

    results = []

    # separa ids válidos e inválidos antes da chamada
    valid_ids = []
    for arxiv_id in id_list:
        if is_valid_arxiv_id(arxiv_id):
            valid_ids.append(arxiv_id)
        else:
            results.append({
                "arxiv_id": arxiv_id,
                "citationCount": None,
                "status": "invalid_arxiv_id"
            })

    # processa só os válidos
    for i in range(0, len(valid_ids), batch_size):
        batch_ids = valid_ids[i:i + batch_size]

        payload = {
            "ids": [f"ARXIV:{arxiv_id}" for arxiv_id in batch_ids]
        }

        try:
            response = requests.post(url, params=params, json=payload, timeout=30)

            if response.status_code == 429:
                print(f"Rate limit no batch {i // batch_size + 1}. Esperando...")
                time.sleep(sleep_seconds)
                response = requests.post(url, params=params, json=payload, timeout=30)

            if response.status_code != 200:
                print(f"Erro no batch {i // batch_size + 1}: {response.status_code}")
                print(response.text)

                for arxiv_id in batch_ids:
                    results.append({
                        "arxiv_id": arxiv_id,
                        "citationCount": None,
                        "status": f"error_{response.status_code}"
                    })
            else:
                data = response.json()

                for arxiv_id, paper in zip(batch_ids, data):
                    if paper:
                        results.append({
                            "arxiv_id": arxiv_id,
                            "citationCount": paper.get("citationCount"),
                            "status": "ok"
                        })
                    else:
                        results.append({
                            "arxiv_id": arxiv_id,
                            "citationCount": None,
                            "status": "not_found"
                        })

        except Exception as e:
            print(f"Exceção no batch {i // batch_size + 1}: {e}")

            for arxiv_id in batch_ids:
                results.append({
                    "arxiv_id": arxiv_id,
                    "citationCount": None,
                    "status": f"exception: {str(e)}"
                })

        time.sleep(sleep_seconds)

    return results

In [22]:
# summary_list = []
# title_list =[]
# id_list =[]
# for paper in papers:
#     summary = paper.get('summary')
#     title = paper.get('title')
#     id = paper.get('arxive_id')
#     summary_list.append(summary)
#     title_list.append(title)
#     id_list.append(id)
    
citation_results = get_citation_counts_batch(id_list=id_list)
citation_results

[{'arxiv_id': '2105.02723', 'citationCount': 116, 'status': 'ok'},
 {'arxiv_id': '2512.19700', 'citationCount': 0, 'status': 'ok'},
 {'arxiv_id': '1806.11202', 'citationCount': 5, 'status': 'ok'},
 {'arxiv_id': '2103.05236', 'citationCount': 32, 'status': 'ok'},
 {'arxiv_id': '2112.05993', 'citationCount': 25, 'status': 'ok'},
 {'arxiv_id': '2307.13365', 'citationCount': 3, 'status': 'ok'},
 {'arxiv_id': '2410.05767', 'citationCount': 2, 'status': 'ok'},
 {'arxiv_id': '2204.00509', 'citationCount': 6, 'status': 'ok'},
 {'arxiv_id': '1802.01528', 'citationCount': 19, 'status': 'ok'},
 {'arxiv_id': '1504.04558', 'citationCount': 56, 'status': 'ok'},
 {'arxiv_id': '2401.08968', 'citationCount': 3, 'status': 'ok'},
 {'arxiv_id': '2509.07225', 'citationCount': 2, 'status': 'ok'},
 {'arxiv_id': '2601.06793', 'citationCount': 0, 'status': 'ok'},
 {'arxiv_id': '2209.10464', 'citationCount': 10, 'status': 'ok'},
 {'arxiv_id': '2412.16166', 'citationCount': 5, 'status': 'ok'},
 {'arxiv_id': '230

In [16]:
citation_map = {
    item["arxiv_id"]: item["citationCount"]
    for item in citation_results
}

for paper in papers:
    arxiv_id = paper.get("arxiv_id")
    paper["citationCount"] = citation_map.get(arxiv_id)


In [17]:
papers

[{'title': 'Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet',
  'authors': ['Luke Melas-Kyriazi'],
  'published': '2021-05-06',
  'url': 'http://arxiv.org/abs/2105.02723v1',
  'summary': 'Do You Even Need Attention? A Stack of Feed-Forward Layers Does\nSurprisingly Well on ImageNet\nLuke Melas-Kyriazi\nOxford University\nlukemk@robots.ox.ac.uk\nAbstract\nThe strong performance of vision transformers on image classiﬁcation and other vision tasks is often attributed\nto the design of their multi-head attention layers. However, the extent to which attention is responsible for this\nstrong performance remains unclear. In this short report,\nwe ask: is the attention layer even necessary? Speciﬁcally,\nwe replace the attention layer in a vision transformer with\na feed-forward layer applied over the patch dimension. The\nresulting architecture is simply a series of feed-forward layers applied over the patch and feature dimensions in an alternatin

### research agend

In [24]:
from dotenv import load_dotenv
load_dotenv()
query = 'What evaluation benchmarks could help measure the distance between current large language models and artificial general intelligence (AGI)'
key_words = key_word(query)
papers = arxiv_search_tool(
    key_words.get("url_to_query"),
    max_results=10
)
result = research_agent(
    query = query ,
    papers = papers,
)

 Keyword Extraction Agent 

Raw output:
{
    "search_phrases": [
        "evaluation benchmarks for large language models",
        "measuring distance between LLMs and AGI",
        "AGI evaluation metrics",
        "benchmarks for artificial general intelligence",
        "LLM performance vs AGI capabilities",
        "AGI progress measurement",
        "language model intelligence assessment",
        "AGI benchmark datasets",
        "evaluating general intelligence in AI",
        "LLM vs AGI comparison metrics"
    ]
}
Starting arXiv request...
arXiv response: HTTP 200, 26748 bytes
[1/10] Metadata Extract concluded
[2/10] Metadata Extract concluded
[3/10] Metadata Extract concluded
[4/10] Metadata Extract concluded
[5/10] Metadata Extract concluded
[6/10] Metadata Extract concluded
[7/10] Metadata Extract concluded
[8/10] Metadata Extract concluded
[9/10] Metadata Extract concluded
[10/10] Metadata Extract concluded
Paper Ranking Agent (research_agent)
creating the final list
ge

In [26]:
import importlib
import src.agents as agents

importlib.reload(agents)

from src.agents import key_word, research_agent

result = research_agent(
    query = query ,
    papers = papers,
)

Paper Ranking Agent (research_agent)
creating the final list
Ranked 10 papers


In [27]:
result

[{'title': 'Levels of AGI for Operationalizing Progress on the Path to AGI',
  'authors': ['Meredith Ringel Morris',
   'Jascha Sohl-Dickstein',
   'Noah Fiedel',
   'Tris Warkentin',
   'Allan Dafoe',
   'Aleksandra Faust',
   'Clement Farabet',
   'Shane Legg'],
  'published': '2023-11-04',
  'url': 'http://arxiv.org/abs/2311.02462v5',
  'summary': 'We propose a framework for classifying the capabilities and behavior of Artificial General Intelligence (AGI) models and their precursors. This framework introduces levels of AGI performance, generality, and autonomy, providing a common language to compare models, assess risks, and measure progress along the path to AGI. To develop our framework, we analyze existing definitions of AGI, and distill six principles that a useful ontology for AGI should satisfy. With these principles in mind, we propose "Levels of AGI" based on depth (performance) and breadth (generality) of capabilities, and reflect on how current systems fit into this ontol